# Fellegi-Sunter Baseline — Manual Validation Notebook

**Purpose.** Pull 5 random record pairs from each predicted tier (`auto_merge`, `human_review`, `no_match`) produced by `fs_splink_baseline` and manually inspect the match probability against the underlying records.

**Where this runs.** This notebook is authored off-VM but **executes only on the VM** against real `MDM_Population` data. Inputs are auto-resolved to the highest-versioned cleaned parquet + candidate-pairs parquet on disk (same convention as `run_real_baseline.py`).

**Output / artifact.** After **Run All**, the reviewer fills in the *Reviewer judgments* section at the bottom of this notebook (per-pair verdict + notes), saves, commits, pushes. **The committed notebook is the written validation record.**

**PHI note.** Output cells will contain identifier values (names, DOB, SSN, addresses). They stay on the VM. If you commit this notebook with outputs, you are committing PHI to the repo — decide deliberately whether to `Cell → All Output → Clear` before committing.

## 1. Setup & imports

In [1]:
from __future__ import annotations

import re
import sys
from datetime import datetime
from pathlib import Path

# Project root = two levels up from notebooks/fellegi_sunter/.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from models.experiments.fs_splink_baseline import fellegi_sunter_baseline as fs
from src.features.blocking import COL_PATID

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = C:\Users\SachinPatel\Desktop\UofC-EMPI


In [2]:
# Notebook-local constants. Override here if you want a different sample size,
# seed, or to point at alternate input directories.
RANDOM_SEED = 42
SAMPLES_PER_TIER = 5
U_MAX_PAIRS = 1e6  # Matches run_real_baseline.py production default.

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BLOCKING_DIR = PROJECT_ROOT / "src" / "features" / "outputs" / "blocking"

TIERS = ["auto_merge", "human_review", "no_match"]

## 2. Auto-resolve highest-versioned inputs

Mirrors the version-resolution convention of `src/data/clean.py` and `src/features/run_blocking.py`: parse the integer `v<N>` token from each filename and pick the maximum.

In [3]:
_VERSION_RE = re.compile(r"_v(\d+)_")


def _latest_versioned(dir_: Path, pattern: str) -> Path:
    """Return the path in `dir_` matching `pattern` with the highest _v<N>_ token."""
    candidates = []
    for p in dir_.glob(pattern):
        m = _VERSION_RE.search(p.name)
        if m:
            candidates.append((int(m.group(1)), p))
    if not candidates:
        raise FileNotFoundError(
            f"No files matching {pattern!r} in {dir_}. "
            "Confirm the cleaning/blocking pipelines have been run on the VM."
        )
    candidates.sort(key=lambda t: t[0])
    return candidates[-1][1]


cleaned_path = _latest_versioned(PROCESSED_DIR, "MDM_Population_cleaned_v*_*.parquet")
pairs_path = _latest_versioned(BLOCKING_DIR, "candidate_pairs_v*_*.parquet")

print(f"Cleaned parquet : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs : {pairs_path.relative_to(PROJECT_ROOT)}")

Cleaned parquet : data\processed\MDM_Population_cleaned_v3_2026_06_11.parquet
Candidate pairs : src\features\outputs\blocking\candidate_pairs_v4_2026_06_11.parquet


## 3. Score with `full_output=True`

`full_output=True` returns the classified frame *before* projection to the 5-col evaluation schema, retaining `match_probability`, `match_weight`, `classification_tier`, `source_blocks`, `n_blocks`, and Splink's `gamma_*` per-field agreement levels for downstream inspection.

Training is the slow step (single call to `run_fs_baseline`).

In [4]:
df_clean = pd.read_parquet(cleaned_path)
print(f"Loaded {len(df_clean):,} cleaned records.")

Loaded 163,364 cleaned records.


In [5]:
df_scored = fs.run_fs_baseline(
    str(pairs_path),
    df_clean,
    u_max_pairs=U_MAX_PAIRS,
    full_output=True,
)

print(f"Scored {len(df_scored):,} candidate pairs.")
print("Tier breakdown:")
print(df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0))

SETTINGS VALIDATION: Errors were identified in your settings dictionary. 

Invalid Columns(s) in Blocking Rule(s)

    SQL: `
EXISTS (
    SELECT 1 FROM candidate_pairs cp
    WHERE cp.PATID_A = l.PATID AND cp.PATID_B = r.PATID
)
`
       - Invalid table names provided (only `l.` and `r.` are valid): `cp.PATID_A`, `cp.PATID_B`

You may want to verify your settings dictionary has valid inputs in all fields before continuing.
You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----
u probability not trained for SSN - Exact match on SSN_clean (comparison vector value: 2). This usually means the comparison level was never observed in the training data.
u probability not trained for Email - Exact match on Email_clean (comparison vector value

Scored 205,424 candidate pairs.
Tier breakdown:
classification_tier
auto_merge       65997
human_review      1303
no_match        138124
Name: count, dtype: int64


## 4. Stratified random sample (5 per tier, fixed seed)

Deterministic across runs given `RANDOM_SEED = 42`. If a tier has fewer than `SAMPLES_PER_TIER` pairs, the notebook takes what's available and prints a warning rather than erroring.

In [6]:
sampled_chunks = []
for tier in TIERS:
    tier_df = df_scored[df_scored["classification_tier"] == tier]
    n_available = len(tier_df)
    n_take = min(SAMPLES_PER_TIER, n_available)
    if n_take < SAMPLES_PER_TIER:
        print(
            f"WARN: tier {tier!r} has only {n_available} pairs; sampling {n_take}."
        )
    sampled_chunks.append(tier_df.sample(n=n_take, random_state=RANDOM_SEED))

sampled = pd.concat(sampled_chunks).reset_index(drop=True)
print(f"Sampled {len(sampled)} pairs total.")
print(sampled["classification_tier"].value_counts().reindex(TIERS, fill_value=0))

Sampled 15 pairs total.
classification_tier
auto_merge      5
human_review    5
no_match        5
Name: count, dtype: int64


## 5. Side-by-side identifier display

For each sampled pair the notebook renders:

1. A one-line header with `classification_tier`, `match_probability`, `match_weight`, and which blocks fired.
2. A two-column DataFrame (`Record A` vs `Record B`) of the cleaned identifier fields. Values are pulled from the cleaned dataframe by `PATID` — this avoids depending on Splink's `_l/_r` suffix convention and shows all fields whether or not the model used them as evidence.
3. A compact view of Splink's `gamma_*` per-field agreement levels (when present), so you can see *why* the model assigned the score it did.

In [7]:
# Fields to display per record, in order. (Label, cleaned-dataframe column name.)
DISPLAY_FIELDS: list[tuple[str, str]] = [
    ("PATID",              "PATID"),
    ("First name",         "FirstNM_clean"),
    ("Middle name",        "MiddleNM_clean"),
    ("Last name",          "LastNM_clean"),
    ("Full name tokens",   "full_name_tokens"),
    ("DOB",                "BirthDT_clean"),
    ("SSN (full)",         "SSN_clean"),
    ("SSN last-4",         "last_4_SSN"),
    ("Email",              "Email_clean"),
    ("Address line 1",     "AddressLine1_clean"),
    ("Address line 2",     "AddressLine2_clean"),
    ("City",               "CityNM_clean"),
    ("State",              "StateCD_clean"),
    ("ZIP",                "ZipCD_clean_base"),
    ("Phones (set)",       "Phones_set"),
]

# Index df_clean by PATID once for O(1) lookups.
_clean_indexed = df_clean.set_index(COL_PATID, drop=False)


def _lookup(patid: str) -> pd.Series:
    """Pull one cleaned record by PATID; returns an empty Series if absent."""
    try:
        return _clean_indexed.loc[patid]
    except KeyError:
        return pd.Series(dtype=object)


def render_identifier_table(patid_a: str, patid_b: str) -> pd.DataFrame:
    """Two-column side-by-side DataFrame of cleaned identifier fields."""
    rec_a = _lookup(patid_a)
    rec_b = _lookup(patid_b)
    rows = {}
    for label, col in DISPLAY_FIELDS:
        if col not in df_clean.columns:
            continue  # field absent in this cleaned parquet version; skip.
        rows[label] = [rec_a.get(col, np.nan), rec_b.get(col, np.nan)]
    return pd.DataFrame.from_dict(
        rows, orient="index", columns=["Record A", "Record B"]
    )


def render_gamma_table(row: pd.Series) -> pd.DataFrame | None:
    """Splink's gamma_<field> per-field agreement levels, if retained."""
    gamma_cols = [c for c in row.index if c.startswith("gamma_")]
    if not gamma_cols:
        return None
    out = pd.DataFrame(
        {"agreement_level": [row[c] for c in gamma_cols]},
        index=[c.removeprefix("gamma_") for c in gamma_cols],
    )
    out.index.name = "comparison"
    return out

In [8]:
from IPython.display import display, Markdown

for i, row in sampled.iterrows():
    pair_num = i + 1
    patid_a = row["PATID_A"]
    patid_b = row["PATID_B"]
    tier = row["classification_tier"]
    p = row.get("match_probability", float("nan"))
    w = row.get("match_weight", float("nan"))
    src_blocks = row.get("source_blocks", "")
    n_blocks = row.get("n_blocks", "")

    display(Markdown(
        f"### Pair {pair_num}/{len(sampled)} \u2014 tier=`{tier}`  \n"
        f"`PATID_A={patid_a}` \u2194 `PATID_B={patid_b}`  \n"
        f"`match_probability={p:.4f}`  \u00b7  `match_weight={w:.3f}`  "
        f"\u00b7  `n_blocks={n_blocks}`  \u00b7  `source_blocks={src_blocks}`"
    ))
    display(render_identifier_table(patid_a, patid_b))
    gammas = render_gamma_table(row)
    if gammas is not None:
        display(Markdown("**Splink agreement levels (`gamma_*`):**"))
        display(gammas)

### Pair 1/15 — tier=`auto_merge`  
`PATID_A=941B2569E4400CA7D79EE3AD05B38450` ↔ `PATID_B=F90BACA6B973388318A513C55DE78C58`  
`match_probability=1.0000`  ·  `match_weight=21.768`  ·  `n_blocks=3`  ·  `source_blocks=B1|B3|B4`

,Record A,Record B
PATID,941B2569E4400CA7D79EE3AD05B38450,F90BACA6B973388318A513C55DE78C58
First name,SAN D,SAN
Middle name,NaN,D
Last name,KYI,KYI
Full name tokens,"['D', 'KYI', 'SAN']","['D', 'KYI', 'SAN']"
DOB,1962-08-07 00:00:00,1962-08-07 00:00:00
SSN (full),699339279,699339279
SSN last-4,9279,9279
Email,sandasr.7kyi@gmail.com,sandar.7kyi@gmail.com
Address line 1,1812 DARROW AVE,6544 N ROCKWELL ST APT 2


**Splink agreement levels (`gamma_*`):**

,agreement_level
comparison,
FirstNM_clean,1
LastNM_clean,4
_dob_str,4
SSN,2
Email,0
Phones_array,0
ZIP,0


### Pair 2/15 — tier=`auto_merge`  
`PATID_A=AEF4FC205C218615D959BA3C6E3D8CCE` ↔ `PATID_B=DA75D0B51F6056247EBD0EA84BD85F6E`  
`match_probability=1.0000`  ·  `match_weight=17.702`  ·  `n_blocks=3`  ·  `source_blocks=B3|B4|B8`

...

,agreement_level
comparison,
FirstNM_clean,0
LastNM_clean,0
_dob_str,0
SSN,-1
Email,-1
Phones_array,1
ZIP,2


### Pair 15/15 — tier=`no_match`  
`PATID_A=0D7ADF3926671D4F5C72E2FA5AF7411B` ↔ `PATID_B=2C63BF046A7E149FCF3D570C0AF62A03`  
`match_probability=0.0000`  ·  `match_weight=-20.836`  ·  `n_blocks=1`  ·  `source_blocks=B5`

,Record A,Record B
PATID,0D7ADF3926671D4F5C72E2FA5AF7411B,2C63BF046A7E149FCF3D570C0AF62A03
First name,LENORE,KIM
Middle name,NaN,NaN
Last name,PENDLETON,PARKER
Full name tokens,"['LENORE', 'PENDLETON']","['KIM', 'PARKER']"
DOB,1957-08-30 00:00:00,1971-05-05 00:00:00
SSN (full),323526999,329728170
SSN last-4,6999,8170
Email,NaN,NaN
Address line 1,4314 S WABASH,4615 N CLINTON


**Splink agreement levels (`gamma_*`):**

,agreement_level
comparison,
FirstNM_clean,0
LastNM_clean,0
_dob_str,0
SSN,0
Email,-1
Phones_array,1
ZIP,1


## 6. Reviewer judgment templates

Run the cell below to generate one markdown template per sampled pair. Copy the printed block into the **Reviewer judgments** markdown cell at the bottom of this notebook and fill in `Reviewer verdict` (`true_match` / `not_match` / `unsure`) and `Reviewer notes` for each pair. Save, commit, push.

Why a printed block (not interactive widgets): markdown survives `nbconvert`, diffs cleanly in git, and the committed notebook is then a self-contained validation record.

In [ ]:
template_lines = []
for i, row in sampled.iterrows():
    pair_num = i + 1
    template_lines.append(
        f"### Pair {pair_num}/{len(sampled)} \u2014 "
        f"PATID_A={row['PATID_A']}, PATID_B={row['PATID_B']}\n"
        f"\n"
        f"- **Predicted tier:** `{row['classification_tier']}`\n"
        f"- **Model match_probability:** {row.get('match_probability', float('nan')):.4f}\n"
        f"- **Reviewer verdict:** [ true_match | not_match | unsure ]\n"
        f"- **Reviewer notes:**\n"
        f"  - \n"
    )

print("\n".join(template_lines))

## 7. Diagnostic summary (non-PHI)

Provenance trail recording which dataset version was validated. Safe to commit even when output cells are cleared.

In [ ]:
tier_counts = df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0)
total = int(tier_counts.sum())

print(f"Executed at         : {datetime.now().isoformat(timespec='seconds')}")
print(f"Cleaned parquet     : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs     : {pairs_path.relative_to(PROJECT_ROOT)}")
print(f"Cleaned records     : {len(df_clean):,}")
print(f"Candidate pairs     : {len(df_scored):,}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Samples per tier    : {SAMPLES_PER_TIER}")
print()
print("Tier distribution (full population):")
for tier in TIERS:
    n = int(tier_counts[tier])
    pct = (n / total * 100) if total else 0.0
    print(f"  {tier:<14s} {n:>10,}  ({pct:5.2f}%)")

## 7.1 Libpostal coverage audit (R2 prerequisite)

Branch-selection gate for the new Address comparison.

`Address_normalized` is produced by libpostal in the cleaning pipeline (`src/data/transformations.py::derive_address_normalized`). When libpostal is not installed, every row's `Address_normalized` is `NaN` and the always-populated `*_clean` siblings (`AddressLine1_clean`, `CityNM_clean`, `StateCD_clean`) must carry the signal instead.

The current FS settings (`build_settings()` post-R2) use **Branch B** — the `*_clean` composite. If libpostal coverage on the VM's cleaned parquet is ≥ 90%, we can swap to **Branch A** (use `Address_normalized` directly with a fuzzy-string comparison). This cell measures the coverage so the decision is data-driven.

In [ ]:
_addr_cols = ["AddressLine1_clean", "CityNM_clean", "StateCD_clean", "Address_normalized"]
_present = [c for c in _addr_cols if c in df_clean.columns]
_missing = [c for c in _addr_cols if c not in df_clean.columns]

print("Address column presence in cleaned parquet:")
for c in _addr_cols:
    flag = "present" if c in df_clean.columns else "MISSING"
    print(f"  {c:<22s} {flag}")

if "Address_normalized" in df_clean.columns:
    _null_rate = df_clean["Address_normalized"].isna().mean()
    _coverage = 1.0 - _null_rate
    print()
    print(f"Address_normalized null rate : {_null_rate:.4f}")
    print(f"Address_normalized coverage  : {_coverage:.4f}")
    print()
    if _coverage >= 0.90:
        print("DECISION → Branch A is viable (libpostal coverage >= 90%).")
        print("           Swap build_settings()'s Address comparison to a")
        print("           hierarchy keyed on Address_normalized.")
    else:
        print("DECISION → Stay on Branch B (the current default).")
        print("           Libpostal coverage is below the 90% threshold; the")
        print("           *_clean composite is the safer signal.")
else:
    print()
    print("Address_normalized is missing entirely — libpostal was not run.")
    print("DECISION → Stay on Branch B (the current default).")

# Non-PHI follow-up: how often do the always-populated *_clean columns line up
# enough for Branch B's levels to fire even without libpostal?
print()
print("*_clean column null rates (Branch B inputs):")
for c in ["AddressLine1_clean", "CityNM_clean", "StateCD_clean"]:
    if c in df_clean.columns:
        print(f"  {c:<22s} null_rate={df_clean[c].isna().mean():.4f}")


## 8. Visual summaries

Population-level views of the baseline's output. All three figures share one palette and styling so they can drop directly into a presentation deck. PNGs are saved to `notebooks/fellegi_sunter/figures/` (gitignored) and filename-versioned by the resolved cleaned-parquet tag so figures don't overwrite across data refreshes.

**Why log scale on the histogram.** Real eMPI score distributions are bimodal: most candidate pairs collapse to ~0 (clear non-matches) or ~1 (clear matches), with a small `human_review` middle band. On a linear y-axis that middle band is visually invisible at production pair counts. Log scale keeps every tier legible without distorting the bimodal shape.

In [ ]:
import re as _re
import matplotlib.pyplot as plt

# ---- Presentation-grade styling (applies to all three figures) -------------
plt.rcParams.update({
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelcolor":    "#333333",
    "axes.edgecolor":     "#555555",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         "#DDDDDD",
    "grid.linewidth":     0.6,
    "grid.alpha":         0.8,
    "xtick.color":        "#333333",
    "ytick.color":        "#333333",
    "legend.frameon":     False,
    "savefig.dpi":        200,
    "savefig.bbox":       "tight",
    "savefig.facecolor":  "white",
})

# Shared palette — aligned with the reference strip chart.
TIER_COLORS = {
    "no_match":     "#B5B5B5",  # neutral gray
    "human_review": "#F0BE7E",  # warm peach
    "auto_merge":   "#88B888",  # muted green
}
TIER_EDGE = {
    "no_match":     "#7F7F7F",
    "human_review": "#C68A3F",
    "auto_merge":   "#4F8A4F",
}
# Display order: low → high match_probability (matches strip chart left→right).
TIER_ORDER_LOW_TO_HIGH = ["no_match", "human_review", "auto_merge"]

# Pull thresholds from the FS module so charts auto-track any retuning.
REVIEW_FLOOR = fs.DEFAULT_REVIEW_FLOOR
AUTO_MERGE_THRESHOLD = fs.DEFAULT_AUTO_MERGE_THRESHOLD

# Output dir for PNGs (gitignored via notebooks/**/figures/).
FIGURES_DIR = Path.cwd() / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Version tag pulled from the resolved cleaned parquet so PNG filenames track
# the dataset version they were generated from.
_version_match = _re.search(r"(v\d+_\d{4}_\d{2}_\d{2})", cleaned_path.name)
VERSION_TAG = _version_match.group(1) if _version_match else "unversioned"

# Pre-compute reusable stats.
_scores = df_scored["match_probability"].to_numpy()
_total = int(len(df_scored))
_tier_counts = df_scored["classification_tier"].value_counts().reindex(
    TIER_ORDER_LOW_TO_HIGH, fill_value=0
).astype(int)
_tier_pct = (_tier_counts / _total * 100) if _total else _tier_counts * 0.0
_five_num = {
    "min":    float(np.min(_scores)),
    "p25":    float(np.quantile(_scores, 0.25)),
    "median": float(np.median(_scores)),
    "p75":    float(np.quantile(_scores, 0.75)),
    "max":    float(np.max(_scores)),
}

print(f"Figures will save to : {FIGURES_DIR.relative_to(PROJECT_ROOT)}")
print(f"Version tag          : {VERSION_TAG}")


### 8.1 Predicted tier breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.2))

y_pos = np.arange(len(TIER_ORDER_LOW_TO_HIGH))
counts = _tier_counts.values
colors = [TIER_COLORS[t] for t in TIER_ORDER_LOW_TO_HIGH]
edges = [TIER_EDGE[t] for t in TIER_ORDER_LOW_TO_HIGH]

bars = ax.barh(y_pos, counts, color=colors, edgecolor=edges, linewidth=1.0, height=0.62)

# Annotate each bar with count + pct, placed just outside the bar end.
_max_count = int(max(counts)) if len(counts) else 0
x_pad = _max_count * 0.012 if _max_count > 0 else 0.5
for bar, tier in zip(bars, TIER_ORDER_LOW_TO_HIGH):
    n = int(_tier_counts[tier])
    pct = float(_tier_pct[tier])
    ax.text(
        bar.get_width() + x_pad,
        bar.get_y() + bar.get_height() / 2,
        f"{n:,}  ({pct:.2f}%)",
        va="center", ha="left", fontsize=10.5, color="#222222",
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(TIER_ORDER_LOW_TO_HIGH, fontsize=11)
ax.invert_yaxis()  # auto_merge on top (top-down reading: highest probability tier first)
ax.set_xlabel("Candidate pairs (count)")
ax.set_xlim(0, _max_count * 1.18 if _max_count > 0 else 1)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)

ax.set_title("Predicted Tier Breakdown", loc="left", pad=18)
ax.text(
    0.0, 1.04,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout()
out_path = FIGURES_DIR / f"tier_breakdown__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Tier counts:")
for tier in TIER_ORDER_LOW_TO_HIGH:
    print(f"  {tier:<14s} {int(_tier_counts[tier]):>10,}  ({float(_tier_pct[tier]):5.2f}%)")


### 8.2 Score distribution — five-number summary vs. classification thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))

# ---- Threshold bands (background) -----------------------------------------
ax.axvspan(0.0, REVIEW_FLOOR, color=TIER_COLORS["no_match"], alpha=0.55, lw=0)
ax.axvspan(REVIEW_FLOOR, AUTO_MERGE_THRESHOLD, color=TIER_COLORS["human_review"], alpha=0.55, lw=0)
ax.axvspan(AUTO_MERGE_THRESHOLD, 1.0, color=TIER_COLORS["auto_merge"], alpha=0.55, lw=0)

# ---- Threshold lines + top labels -----------------------------------------
for x, label in [(REVIEW_FLOOR, "review_floor"), (AUTO_MERGE_THRESHOLD, "auto_merge")]:
    ax.axvline(x, color="#555555", linestyle="--", linewidth=1.1)
    ax.text(x, 1.02, label, transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=10, color="#444444")
    ax.text(x, 0.96, f"{x:.2f}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9.5, color="#444444")

# ---- Tier % labels centered inside each band ------------------------------
band_centers = [
    ("no_match",     REVIEW_FLOOR / 2),
    ("human_review", (REVIEW_FLOOR + AUTO_MERGE_THRESHOLD) / 2),
    ("auto_merge",   (AUTO_MERGE_THRESHOLD + 1.0) / 2),
]
for tier, x_center in band_centers:
    ax.text(x_center, 0.28, tier, ha="center", va="center",
            fontsize=11, fontweight="semibold", color="#333333")
    ax.text(x_center, 0.14, f"({float(_tier_pct[tier]):.1f}%)",
            ha="center", va="center", fontsize=10, color="#555555")

# ---- Five-number summary dots + labels ------------------------------------
# Upper track keeps dots clear of band labels below.
dot_y = 0.70

# Handle the common case where min==p25 (both 0) or max==p75 (both 1) by
# stacking the two collided labels vertically so they don't overprint.
_left_collide = abs(_five_num["min"] - _five_num["p25"]) < 1e-6
_right_collide = abs(_five_num["max"] - _five_num["p75"]) < 1e-6

ax.scatter([_five_num["min"]], [dot_y], s=46, color="#111111", zorder=5)
if _left_collide:
    ax.text(_five_num["min"] + 0.012, dot_y + 0.08,
            f"min={_five_num['min']:g}", ha="left", va="bottom",
            fontsize=9.5, color="#222222")
    ax.text(_five_num["p25"] + 0.012, dot_y - 0.08,
            f"p25={_five_num['p25']:g}", ha="left", va="top",
            fontsize=9.5, color="#222222")
else:
    ax.text(_five_num["min"] + 0.012, dot_y,
            f"min={_five_num['min']:g}", ha="left", va="center",
            fontsize=9.5, color="#222222")
    ax.scatter([_five_num["p25"]], [dot_y], s=46, color="#111111", zorder=5)
    ax.text(_five_num["p25"] + 0.012, dot_y,
            f"p25={_five_num['p25']:g}", ha="left", va="center",
            fontsize=9.5, color="#222222")

# Median (always its own dot; label above to avoid the dot itself).
ax.scatter([_five_num["median"]], [dot_y], s=46, color="#111111", zorder=5)
ax.text(_five_num["median"], dot_y + 0.10,
        f"median\n{_five_num['median']:.4f}",
        ha="center", va="bottom", fontsize=9.5, color="#222222")

ax.scatter([_five_num["max"]], [dot_y], s=46, color="#111111", zorder=5)
if _right_collide:
    ax.text(_five_num["max"] - 0.012, dot_y + 0.08,
            f"max={_five_num['max']:g}", ha="right", va="bottom",
            fontsize=9.5, color="#222222")
    ax.text(_five_num["p75"] - 0.012, dot_y - 0.08,
            f"p75={_five_num['p75']:g}", ha="right", va="top",
            fontsize=9.5, color="#222222")
else:
    ax.text(_five_num["max"] - 0.012, dot_y,
            f"max={_five_num['max']:g}", ha="right", va="center",
            fontsize=9.5, color="#222222")
    ax.scatter([_five_num["p75"]], [dot_y], s=46, color="#111111", zorder=5)
    ax.text(_five_num["p75"] - 0.012, dot_y,
            f"p75={_five_num['p75']:g}", ha="right", va="center",
            fontsize=9.5, color="#222222")

# ---- Axes cosmetics -------------------------------------------------------
ax.set_xlim(-0.005, 1.005)
ax.set_ylim(0, 1)
ax.set_xlabel("match_probability (score)")
ax.set_yticks([])
ax.grid(False)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)

ax.set_title(
    "Score Distribution — Five-Number Summary vs. Classification Thresholds",
    loc="left", pad=22,
)
ax.text(
    0.0, 1.13,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout()
out_path = FIGURES_DIR / f"score_distribution_strip__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Five-number summary of match_probability:")
for k, v in _five_num.items():
    print(f"  {k:<6s} {v:.6f}")


### 8.3 Score histogram — match_probability colored by tier

In [ ]:
HIST_BINS = 50

fig, ax = plt.subplots(figsize=(11, 4.6))

bin_edges = np.linspace(0.0, 1.0, HIST_BINS + 1)
bin_width = bin_edges[1] - bin_edges[0]
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Build per-tier histograms over identical bins, then stack.
tier_hists = {}
for tier in TIER_ORDER_LOW_TO_HIGH:
    scores_t = df_scored.loc[df_scored["classification_tier"] == tier, "match_probability"].to_numpy()
    h, _ = np.histogram(scores_t, bins=bin_edges)
    tier_hists[tier] = h

cumulative = np.zeros(HIST_BINS, dtype=float)
for tier in TIER_ORDER_LOW_TO_HIGH:
    h = tier_hists[tier]
    ax.bar(
        bin_centers, h, width=bin_width * 0.95,
        bottom=cumulative,
        color=TIER_COLORS[tier], edgecolor=TIER_EDGE[tier], linewidth=0.5,
        label=f"{tier}  ({int(_tier_counts[tier]):,})",
        align="center",
    )
    cumulative = cumulative + h

# Log scale with a small floor so empty bins don't visually clip.
ax.set_yscale("log")
_max_total = max(cumulative.max(), 1)
ax.set_ylim(0.5, _max_total * 3)

# Threshold lines + top labels.
for x, label in [(REVIEW_FLOOR, "review_floor"), (AUTO_MERGE_THRESHOLD, "auto_merge")]:
    ax.axvline(x, color="#444444", linestyle="--", linewidth=1.1, zorder=3)
    ax.text(x, 1.02, f"{label} = {x:.2f}", transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=10, color="#444444")

ax.set_xlim(-0.01, 1.01)
ax.set_xlabel("match_probability (score)")
ax.set_ylabel("Candidate pairs (count, log scale)")
ax.grid(axis="x", visible=False)
ax.grid(axis="y", visible=False)

ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3,
    fontsize=10, handlelength=1.4, handleheight=1.0, borderpad=0.6,
)

ax.set_title(
    "Score Histogram — match_probability by Tier",
    loc="left", y=1.18, pad=0,
)
ax.text(
    0.0, 1.10,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs  ·  {HIST_BINS} bins",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout(rect=[0, 0, 1, 0.88])
out_path = FIGURES_DIR / f"score_histogram__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Five-number summary of match_probability:")
for k, v in _five_num.items():
    print(f"  {k:<6s} {v:.6f}")
print()
print("Per-tier counts (legend mirrors these):")
for tier in TIER_ORDER_LOW_TO_HIGH:
    print(f"  {tier:<14s} {int(_tier_counts[tier]):>10,}  ({float(_tier_pct[tier]):5.2f}%)")


## 9. Threshold-band sampling deep-dives

Two targeted sampling passes that feed the threshold-tuning conversation (separate from the Section-5 stratified sample, which served the broader sanity-check goal):

- **§9.1** — A 20-pair random sample drawn from the `human_review` tier only, to characterize what kinds of identifier discrepancies land in that middle band.
- **§9.2** — A 12-pair-per-band stratified sample drawn from the score bands surrounding each tier boundary (`[0.45, 0.55)` around `review_floor`, `[0.85, 0.95)` around `auto_merge`), to probe whether the current thresholds are well placed.

Both sections render the same identifier-table + `gamma_*` view as Section 5 and emit reviewer-judgment templates at the end so a manual reviewer can record verdicts in-line. Pairs already shown in §9.1 are deduplicated out of §9.2.

### 9.1 Human-review band characterization (20-pair random sample)

Goal: characterize what kinds of pair-discrepancies actually land in `human_review` (the small middle band — 1,369 pairs on the v3/v4 baseline, ~0.67% of all scored pairs). A 20-pair sample is enough to spot dominant failure modes without committing to a full census.

Reuses `render_identifier_table()` and `render_gamma_table()` from Section 5 so the rendered output is identical in shape to the 15-pair stratified sample reviewed in Section 5 — just sourced from `human_review` only.

In [ ]:
# §9.1 human_review band — random 20-pair sample for hand review.
PHASE4_HR_SAMPLE_N = 20

_hr_pool = df_scored[df_scored["classification_tier"] == "human_review"]
_hr_take = min(PHASE4_HR_SAMPLE_N, len(_hr_pool))
if _hr_take < PHASE4_HR_SAMPLE_N:
    print(f"WARN: human_review has only {len(_hr_pool)} pairs; sampling {_hr_take}.")
_hr_sample = _hr_pool.sample(n=_hr_take, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"§9.1 human_review sample: {len(_hr_sample)} pairs (seed={RANDOM_SEED})")
print()

for i, row in _hr_sample.iterrows():
    patid_a, patid_b = row["PATID_A"], row["PATID_B"]
    p = row.get("match_probability", float("nan"))
    w = row.get("match_weight", float("nan"))
    src_blocks = row.get("source_blocks", "")
    n_blocks = row.get("n_blocks", "")
    display(Markdown(
        f"#### Pair {i+1}/{len(_hr_sample)} — tier=`{row['classification_tier']}`  \n"
        f"`PATID_A={patid_a}` ↔ `PATID_B={patid_b}`  \n"
        f"`match_probability={p:.4f}`  ·  `match_weight={w:.3f}`  ·  "
        f"`n_blocks={n_blocks}`  ·  `source_blocks={src_blocks}`"
    ))
    display(render_identifier_table(patid_a, patid_b))
    gammas = render_gamma_table(row)
    if gammas is not None:
        display(gammas)


### 9.2 Threshold-boundary stratified samples (12 per band)

Two bands:

- `review_floor_band` — `score ∈ [0.45, 0.55)`: pairs straddling the `no_match` ↔ `human_review` threshold.
- `auto_merge_band` — `score ∈ [0.85, 0.95)`: pairs straddling the `human_review` ↔ `auto_merge` threshold.

Goal: judge by hand whether borderline pairs look more like the tier above or below the current threshold. Drives the decision on whether to retune `DEFAULT_AUTO_MERGE_THRESHOLD` (0.90) or `DEFAULT_REVIEW_FLOOR` (0.50).

Pairs already shown in §9.1 are deduplicated out of this sample so a reviewer never sees the same pair twice.

In [10]:
# §9.2 stratified threshold-boundary sample (10-15 per band, dedup against §9.1).
PHASE4_PER_BAND = 12  # mid-point of the requested 10-15

_phase4_seen = set(zip(_hr_sample["PATID_A"], _hr_sample["PATID_B"])) if "_hr_sample" in dir() else set()
_scored_avail = df_scored[~df_scored.set_index(["PATID_A", "PATID_B"]).index.isin(_phase4_seen)]

def _band_sample(low: float, high: float) -> pd.DataFrame:
    mask = (_scored_avail["match_probability"] >= low) & (_scored_avail["match_probability"] < high)
    band = _scored_avail.loc[mask]
    n_take = min(PHASE4_PER_BAND, len(band))
    if n_take < PHASE4_PER_BAND:
        print(f"WARN: band [{low}, {high}) has only {len(band)} pairs; sampling {n_take}.")
    return band.sample(n=n_take, random_state=RANDOM_SEED).reset_index(drop=True)

_floor_sample = _band_sample(0.45, 0.55)
_auto_sample = _band_sample(0.85, 0.95)

print(f"§9.2 review_floor_band [0.45, 0.55): {len(_floor_sample)} pairs")
print(f"§9.2 auto_merge_band  [0.85, 0.95): {len(_auto_sample)} pairs")
print()

for _label, _frame in [
    ("review_floor_band [0.45, 0.55)", _floor_sample),
    ("auto_merge_band [0.85, 0.95)", _auto_sample),
]:
    display(Markdown(f"### §9.2 — {_label}"))
    for i, row in _frame.iterrows():
        patid_a, patid_b = row["PATID_A"], row["PATID_B"]
        p = row.get("match_probability", float("nan"))
        w = row.get("match_weight", float("nan"))
        src_blocks = row.get("source_blocks", "")
        display(Markdown(
            f"#### Pair {i+1}/{len(_frame)} — tier=`{row['classification_tier']}`  \n"
            f"`PATID_A={patid_a}` ↔ `PATID_B={patid_b}`  \n"
            f"`match_probability={p:.4f}`  ·  `match_weight={w:.3f}`  ·  "
            f"`source_blocks={src_blocks}`"
        ))
        display(render_identifier_table(patid_a, patid_b))
        gammas = render_gamma_table(row)
        if gammas is not None:
            display(gammas)


§9.2 review_floor_band [0.45, 0.55): 12 pairs
§9.2 auto_merge_band  [0.85, 0.95): 12 pairs



### §9.2 — review_floor_band [0.45, 0.55)

#### Pair 1/12 — tier=`no_match`  
`PATID_A=61D14D4FAB4072EBB6797AE46E9E8680` ↔ `PATID_B=E02F33DAC931CBF9865CEC004474BFA1`  
`match_probability=0.4733`  ·  `match_weight=-0.154`  ·  `source_blocks=B5|B6`

,Record A,Record B
PATID,61D14D4FAB4072EBB6797AE46E9E8680,E02F33DAC931CBF9865CEC004474BFA1
First name,GERARDO,ASHLY
Middle name,NaN,NaN
Last name,FRANCISCO MENDOZA,MADRID SUCHITE
Full name tokens,"['FRANCISCO', 'GERARDO', 'MENDOZA']","['ASHLY', 'MADRID', 'SUCHITE']"
DOB,2009-10-09 00:00:00,2011-01-04 00:00:00
SSN (full),NaN,NaN
SSN last-4,NaN,NaN
Email,pcruz@nyap.org,pcruz@nyap.org
Address line 1,2435 W DIVISION ST,2435 W DIVISION ST


,agreement_level
comparison,
FirstNM_clean,0
LastNM_clean,0
_dob_str,0
SSN,-1
Email,2
Phones_array,1
ZIP,2


...

,Record A,Record B
PATID,00A346E3452250650488DCB33D563985,06A61F36FBF53F381F214FACE8B56BA7
First name,LOOKMAN,LUKUMAN
Middle name,A,NaN
Last name,MUHAMMED,MUHAMMED
Full name tokens,"['A', 'LOOKMAN', 'MUHAMMED']","['LUKUMAN', 'MUHAMMED']"
DOB,1993-11-22 00:00:00,1993-11-22 00:00:00
SSN (full),337961145,NaN
SSN last-4,1145,NaN
Email,NaN,NaN
Address line 1,4506 N SHERIDAN,1717 W NORTHSHORE


,agreement_level
comparison,
FirstNM_clean,0
LastNM_clean,4
_dob_str,4
SSN,-1
Email,-1
Phones_array,0
ZIP,1


#### Pair 12/12 — tier=`auto_merge`  
`PATID_A=95D1836F915B9C831DF358C16A24D3F9` ↔ `PATID_B=ED3E769D0FF743DCBC66BCEEE31C3171`  
`match_probability=0.9392`  ·  `match_weight=3.949`  ·  `source_blocks=B3`

,Record A,Record B
PATID,95D1836F915B9C831DF358C16A24D3F9,ED3E769D0FF743DCBC66BCEEE31C3171
First name,CATALINA,CATALINA
Middle name,NaN,NaN
Last name,QUIROZ,GARCIA
Full name tokens,"['CATALINA', 'QUIROZ']","['CATALINA', 'GARCIA']"
DOB,1957-04-30 00:00:00,1957-04-30 00:00:00
SSN (full),NaN,NaN
SSN last-4,NaN,NaN
Email,NaN,NaN
Address line 1,3438 W BEACH AVE,2728 S KOLIN AVE


,agreement_level
comparison,
FirstNM_clean,3
LastNM_clean,0
_dob_str,4
SSN,-1
Email,-1
Phones_array,0
ZIP,1


In [ ]:
# Reviewer-judgment templates for the §9 samples. Copy the printed block into
# the Reviewer judgments markdown cell at the bottom of this notebook, beneath
# the Section-6 templates.

_phase4_lines: list[str] = []
_phase4_lines.append("<!-- §9 sampling judgments -->\n")
for _label, _frame in [
    ("§9.1 human_review", _hr_sample),
    ("§9.2 review_floor_band", _floor_sample),
    ("§9.2 auto_merge_band", _auto_sample),
]:
    if _frame is None or _frame.empty:
        continue
    _phase4_lines.append(f"\n#### {_label} judgments\n")
    for _i, _row in _frame.iterrows():
        _phase4_lines.append(
            f"- PATID_A={_row['PATID_A']}, PATID_B={_row['PATID_B']}, "
            f"score={_row.get('match_probability', float('nan')):.4f}, "
            f"tier=`{_row['classification_tier']}`  \n"
            f"  - **Reviewer verdict:** [ true_match | not_match | unsure ]  \n"
            f"  - **Reviewer notes:**  \n"
        )

print("".join(_phase4_lines))


## Reviewer judgments

 - Pair: PATID_A=<682F05B6FC0B423E4BAF2E65973EE907> PATID_B=<E6B86E52BF368B579B5D7C1AF47F3AF6> | section=§9.1 | score=<0.7406>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + exact address + same phone set>
    notes: <no additional notes to report>


- Pair: PATID_A=<AABECF0FB6679FD22AB032966012FFEB> PATID_B=<F615484F791DDE244D9F07BBEEA32761> | section=§9.1 | score=<0.7294>
    verdict: <different>
    confidence: <medium>
    pattern: <roomate-same-address>
    address: <same household>
    drivers: <one line>
    notes: <The address is technically exact however one patient record has spaces in between the street information whereas the other one does not>

- Pair: PATID_A=<70157C3E98BE59DEAE4A984BAB7559A9> PATID_B=<9F2BE9307A372B4758DAD8383CA30080> | section=§9.1 | score=<0.5677>
    verdict: <different>
    confidence: <high>
    pattern: <namesake-same-name>
    address: <different>
    drivers: <different first name, same last name, same DOB, different address, different phone set>
    notes: <The two records only matched on last name and DOB but had completely different addresses, zipcodes, phone numbers, and first names which was odd>


- Pair: PATID_A=<1CF523B0FFB944EDA2F6CAE741C8B72E> PATID_B=<97732CBC73F85505A764A5765D969913> | section=§9.1 | score=<0.8830>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + exact address + same email + different DOB>
    notes: <no additional notes to report>


- Pair: PATID_A=<378FA81D4D91666C9278E28505A38BCC> PATID_B=<C6A30F008554A9D97B9C1FB1AC132825> | section=§9.1 | score=<0.7800>
    verdict: <unsure>
    confidence: <low>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <other>
    notes: <The data is odd because both records match apart from the last name and date of birth. Originally I had thought it was the same person but just a name change due to marriage but that then the date of birth would have been the same so final verdice is unsure.>

- Pair: PATID_A=<7B2EF72A421AE6A58CA6366D41C1A1CA> PATID_B=<D90C5275BFFEB04D378E5182694623B8> | section=§9.1 | score=<0.6194>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <unusable>
    drivers: <same last name + same phone number>
    notes: <The address fields for one of the PATID_A are 'NA' so unclear if these records are from patients that live in the same household but just different individuals>

- Pair: PATID_A=<1EC8E1BEA622557873D992BF00D7C507> PATID_B=<9570F442CDA5997423B3B023755F14E3> | section=§9.1 | score=<0.6514>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<291CFBB3004F9A91F97ADED63845C58A> PATID_B=<D0AD271E455D6E7C23D748BAC472F949> | section=§9.1 | score=<0.8631>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<0949BC832B61AEB6F749F62D21C26812> PATID_B=<1D8E93DBB01A5497A0394F9DD90179F2> | section=§9.1 | score=<0.6500>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<081D634EF5668AE1DED0F305CE8383B3> PATID_B=<36DF6F5E0272DD9F757416F9694A28A8> | section=§9.1 | score=<0.7753>
    verdict: <unsure>
    confidence: <medium>
    pattern: <other>
    address: <exact>
    drivers: <different last name + different DOB + same first name + exact address + same phone number>
    notes: <the data is odd because the records match on first name but have different last name + different DOB but same exact address and phone number. It is also difficult to determine the verdict because PATID_A has 'NA' in Social security field.>

- Pair: PATID_A=<BC977805B1B9E6CF6661DB12F0BA3F5B> PATID_B=<CC812C5D2CCAD8BBFD55CC3626572AF9> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <medium>
    pattern: <roommate-same-household>
    address: <exact>
    drivers: <different first name + different last name + different DOB but exact address and phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<16E2E01F3491DA321A0A6080C91004D5> PATID_B=<C04DD69888E0B68D8B18400239714D00> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <same-household>
    drivers: <different first name, same-ish last name, same household, same phone number, same email >
    notes: <the last name for PATID_B has an additional name in it however there is enough overlap to suggest it is still the same last name shared by the two records>
   
- Pair: PATID_A=<266A55B6F2DA2F18D726E349E36E056F> PATID_B=<DFF9ADD06147B852E27EF51C22F02ECA> | section=§9.1 | score=<0.5290>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same last name + different first name + different DOB + different address + same phone number>
    notes: <The last name is shared between the two records and the same city + state + zipcode however the AL1 differs. This suggests potentially same family but live in different addresses now.>

- Pair: PATID_A=<450479ED367FC02CD538DF431FD722F4> PATID_B=<7D96EB724CFFDC20E010ED9A670A2E41> | section=§9.1 | score=<0.8607>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same last name + different first name + different DOB + different address + same phone number>
    notes: <The last name is shared between the two records and the same city + state + zipcode however the AL1 differs. This suggests potentially same family but live in different addresses now.>

- Pair: PATID_A=<28DECA1758F0E1CEE8374E33B366009E> PATID_B=<65AEA6D9C8DC70E66A9C53915A1B2AD6> | section=§9.1 | score=<0.5090>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<9B8F2F432876EC6D2805C79510FE7A8B> PATID_B=<F4644B8B7EDE7B3097568C1920170C1C> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <medium>
    pattern: <roommate-same-address>
    address: <exact>
    drivers: <different first name + different last name + different DOB + exact address + same email + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<4F1228261B37E2663D85684B6F5F23BF> PATID_B=<8E6BF0201F5EDC75DB73C66299643901> | section=§9.1 | score=<0.8516>
    verdict: <different>
    confidence: <low>
    pattern: <other>
    address: <different>
    drivers: <same last name + same DOB + different address + different phone number>
    notes: <PATID_B almost has the same first name as PATID_A + same last name but different DOB and do not share the same address or phone number. Final verdict is I think they are different patients however my confidence is low.>

- Pair: PATID_A=<41A4634D5B113A0298C4DF66162D92B2> PATID_B=<99A27B3C403CD07CDC90DF44B19F2ED2> | section=§9.1 | score=<0.8631>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<816582D9813E6B3787A223DEEB480B65> PATID_B=<F087ECC809544465E28DE6F4DCDC3023> | section=§9.1 | score=<0.6156>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<15FA264E76C3952E947781D288ED28E4> PATID_B=<E01448160E24677A9B8BFD9428765F89> | section=§9.1 | score=<0.7110>
    verdict: <unsure>
    confidence: <low>
    pattern: <other>
    address: <exact>
    drivers: <same first name + different last name + different DOB + exact address + same phone number>
    notes: <What is odd is these patient records have the same first name, same address and same phone number but are different in last name and DOB which would suggest they are different patients. Unable to verify on SSN because both patient records do not contain the SSN>

- Pair: PATID_A=<61D14D4FAB4072EBB6797AE46E9E8680> PATID_B=<E02F33DAC931CBF9865CEC004474BFA1> | section=§9.1 | score=<0.4733>
    verdict: <different>
    confidence: <high>
    pattern: <roommate-same-address>
    address: <exact>
    drivers: <different first name + different last name + same email + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<0FEEF5F76952F03CD2CC22AB60832A6C> PATID_B=<C9BA88D6EC666A621A1AE2FA85A67E64> | section=§9.1 | score=<0.4695>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>
   
- Pair: PATID_A=<B4D2F15EE6438D40BC025DAE43F934FD> PATID_B=<D93D21F1040898156C0E604B5AEED4AD> | section=§9.1 | score=<0.5360>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <same last name + same city + state + zip + different AL1>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<9C7079700F7E4642C9A6CB094C873B7F> PATID_B=<F7D8641948A22C485BD1A4DE17CB87C5> | section=§9.1 | score=<0.4958>
    verdict: <unsure>
    confidence: <medium>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <same first name + last name + almost same DOB + different AL1 + same city + state + zipcode + different phone number>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <The data is odd because the patient records have the same first name, last name, and the DOB only differs by one number in the month which could suggest a typo>

- Pair: PATID_A=<EF63FD2E24E598A9831899F9E7521EDA> PATID_B=<F5962686BD4EA1EE33368D543ED0F177> | section=§9.1 | score=<0.5487>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + different DOB + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<B3DA94C0BAD0F527AA64D2B680453EB2> PATID_B=<DB604069B623F8A94D8F32E47D580789> | section=§9.1 | score=<0.4570>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <same-household>
    drivers: <different first name + different last name + different DOB + same address + same phone number>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<73AD38C41B03EC774E70F44B94118767> PATID_B=<B0D4A5C82A7D4D2DC22E357BF1388D9B> | section=§9.1 | score=<0.4916>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>
   
- Pair: PATID_A=<4878430CF7F6D3E1136BAED7C60F539E> PATID_B=<FDD50488B5444FC49A755CD3F29C2236> | section=§9.1 | score=<0.4550>
    verdict: <unsure>
    confidence: <low>
    pattern: <same-person-name-change>
    address: <different>
    drivers: <same last name + same DOB + different address + different phone number>
    threshold (§9.2 only): <model-correct>
    notes: <the data is odd because the first name differs by one letter which suggests a typo given that the last names are the same and the DOB are the same>

- Pair: PATID_A=<1FBBB6AD96D0B386C5616BD4DA669179> PATID_B=<90266ACAA98E5DFDD1F6AE8B894D70E2> | section=§9.1 | score=<0.5437>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same email + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<6D7A610F463F4A27C8B496956905E288> PATID_B=<DB9FDAAC432C177B923691A172240204> | section=§9.1 | score=<0.5258>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<A6A60A32B9C166AF79D62D20976CB9D1> PATID_B=<F7D8641948A22C485BD1A4DE17CB87C5> | section=§9.1 | score=<0.4958>
    verdict: <same>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <same-city-state-zip>
    drivers: <same first name + same last name + same city + state + zipcode>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <The DOB for the patient records differs by one digit in the month which could suggest a typo given that the first name and last name are the same>

- Pair: PATID_A=<3970FB9B7DB4610EFA41BB0742BC3F2A> PATID_B=<D6A64CCCF21D81E3A4F28D23629FBB2B> | section=§9.1 | score=<0.5157>
    verdict: <same>
    confidence: <low>
    pattern: <same person name change>
    address: <exact>
    drivers: <same first name + different last name + exact address + same phone + different DOB>
    threshold (§9.2 only): <model-correct | should-be-higher-tier | should-be-lower-tier>
    notes: <the difference in DOB could suggest they are different patients so my confidence is low>

- Pair: PATID_A=<4647BF4D0A5ABB5DEF5DBD3545C062D5> PATID_B=<C3258D1EC2E01D6F3147E6836A4E56A1> | section=§9.1 | score=<0.9153>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<2A9E805B0E925849D85C06910D161140> PATID_B=<552AD9E5304E308AC243EEC20FA314D7> | section=§9.1 | score=<0.9309>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same first name + same last name + same DOB + different SSN>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <The model made a critical error here - if there is a difference in SSN this should be an automatic no match >

- Pair: PATID_A=<82B78923F1585C3D7B4B0DD6E85F5552> PATID_B=<9672BE0C5B5A917D43ED51CB6BA48207> | section=§9.1 | score=<0.8541>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<A585399515E09F9482B9C8465B065783> PATID_B=<DC12DC8D542163619B9910EB59E28543> | section=§9.1 | score=<0.8937>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<00A75C5C27F1E9066A5C18D5FC7D557B> PATID_B=<D578816BF1D9EBD00FE64BF7434A5DC5> | section=§9.1 | score=<0.8721>
    verdict: <unsure>
    confidence: <medium>
    pattern: <roomates-same-address>
    address: <exact>
    drivers: <same first name + different last name + exact address + different DOB + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<BF69FBAEEAB134B1B619D0E6A18EAC6C> PATID_B=<D37991416B5D345FB781F2C7F5EEDFA1> | section=§9.1 | score=<0.8824>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <same first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<1619504FF993089082820E778DD5935E> PATID_B=<52AE54209B94D7E37F6250617409DBA7> | section=§9.1 | score=<0.9409>
    verdict: <different>
    confidence: <high>>
    pattern: <other>
    address: <different>
    drivers: <different first name + same last name + different DOB + different AL1 + same city + state + zip + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<4F109F3C53C518D771F8E84AE03BE003> PATID_B=<CC9C232A6F3CEDE1B0B3868115D7FE8F> | section=§9.1 | score=<0.8526>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <different>
    drivers: <same first name + different last name +same DOB + different address + different phone number >
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<5E44CA62C42EFA5BE9579AEAEA5C5557> PATID_B=<CE548FF534B46CD262103AFFAEA6D8B7> | section=§9.1 | score=<0.9294>
    verdict: <different>
    confidence: <medium>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + different DOB + different AL1 + same city + state + zip + different phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<54256AE52D2F6A9E7B3587407CE8437D> PATID_B=<6E764354E5CFAA4C860F6AD8378DAB88> | section=§9.1 | score=<0.9439>
    verdict: <same>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <different first name + same last name + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<00A346E3452250650488DCB33D563985> PATID_B=<06A61F36FBF53F381F214FACE8B56BA7> | section=§9.1 | score=<0.9311>
    verdict: <unsure>
    confidence: <high>
    pattern: <namesake-same-name>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + same DOB + different AL1 + same city + state + zip + different phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<95D1836F915B9C831DF358C16A24D3F9> PATID_B=<ED3E769D0FF743DCBC66BCEEE31C3171> | section=§9.1 | score=<0.9392>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <same-city-state-zip>
    drivers: <same first name + different last name + same DOB >
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

